Code to replicate cross-border climate risk analysis, with EORA26 as MRIOT and ND-GAIN as climate risk index.

# MRIOT
----

ICIO

In [2]:
import pymrio
oecd_storage = "MRIOT/ICIO/ICIO2021_2018.zip"
oecd_file = pymrio.parse_oecd(path=oecd_storage)

oecd_file.rename_sectors(    {"01T02":	"Agriculture, hunting, forestry",
"03":	"Fishing and aquaculture",
"05T06":	"Mining and quarrying, energy producing products",
"07T08":	"Mining and quarrying, non-energy producing products",
"09":	"Mining support service activities",
"10T12":	"Food products, beverages and tobacco",
"13T15":	"Textiles, textile products, leather and footwear",
"16":	"Wood and products of wood and cork",
"17T18":	"Paper products and printing",
"19":	"Coke and refined petroleum products",
"20":	"Chemical and chemical products",
"21":	"Pharmaceuticals, medicinal chemical and botanical products",
"22":	"Rubber and plastics products",
"23":	"Other non-metallic mineral products",
"24":	"Basic metals",
"25":	"Fabricated metal products",
"26":	"Computer, electronic and optical equipment",
"27":	"Electrical equipment",
"28":	"Machinery and equipment, nec ",
"29":	"Motor vehicles, trailers and semi-trailers",
"30":	"Other transport equipment",
"31T33":	"Manufacturing nec; repair and installation of machinery and equipment",
"35":	"Electricity, gas, steam and air conditioning supply",
"36T39":	"Water supply; sewerage, waste management and remediation activities",
"41T43":	"Construction",
"45T47":	"Wholesale and retail trade; repair of motor vehicles",
"49":	"Land transport and transport via pipelines",
"50":	"Water transport",
"51":	"Air transport",
"52":	"Warehousing and support activities for transportation",
"53":	"Postal and courier activities",
"55T56":	"Accommodation and food service activities",
"58T60":	"Publishing, audiovisual and broadcasting activities",
"61":	"Telecommunications",
"62T63":	"IT and other information services",
"64T66":	"Financial and insurance activities",
"68":	"Real estate activities",
"69T75":	"Professional, scientific and technical activities",
"77T82":	"Administrative and support services",
"84":	"Public administration and defence; compulsory social security",
"85":	"Education",
"86T88":	"Human health and social work activities",
"90T93":	"Arts, entertainment and recreation",
"94T96":	"Other service activities",
"97T98":	"Activities of households as employers; undifferentiated goods- and services-producing activities of households for own use"
})

oecd_file.calc_all()
mriot = oecd_file

c:\Users\delah\anaconda3\envs\Boario\lib\site-packages\pymrio\tools\ioparser.py:1671: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  Z.loc[co_name, :] + Z.loc[agg_list, :].groupby(level="sector", axis=0).sum()
c:\Users\delah\anaconda3\envs\Boario\lib\site-packages\pymrio\tools\ioparser.py:1675: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  Y.loc[co_name, :] + Y.loc[agg_list, :].groupby(level="sector", axis=0).sum()
c:\Users\delah\anaconda3\envs\Boario\lib\site-packages\pymrio\tools\ioparser.py:1681: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  Z.loc[:, co_name] + Z.loc[:, agg_list].groupby(level="sector", axis=1).sum()
c:\Users\delah\anaconda3\envs\Boario\lib\site-packages\pymrio\tools\ioparser.py:1687: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` witho

# Climate risk index
-----

## ND-GAIN

The latest version of ND-Gain (as 13/03/2024) is up to 2021, but is estimated based on future climate change projection.

Initially, there are 185 countries in ND-Gain and 66+1 in ICIO.

In [3]:
import pandas as pd
import numpy as np

# Importing ND-Gain 
########

file_path = 'ND_GAIN/resources/gain/gain.csv'
local_climate_risk = pd.read_csv(file_path)
# Selecting only the required columns and renaming them
local_climate_risk = local_climate_risk[['ISO3', '2021']]
local_climate_risk.columns = ['Country', 'Local Climate Risk']
local_climate_risk = local_climate_risk.dropna(subset=['Local Climate Risk']).reset_index(drop=True)
local_climate_risk = local_climate_risk.sort_values(by='Local Climate Risk', ascending=False)


# Find the row with the 'SDN' code
sdn_row = local_climate_risk[local_climate_risk['Country'] == 'SDN']

# Create new rows for 'SDS' and 'SUD' with the same data as 'SDN'
sds_row = sdn_row.copy()
sds_row['Country'] = 'SDS'
sud_row = sdn_row.copy()
sud_row['Country'] = 'SUD'

# Concatenate new rows to the DataFrame
local_climate_risk = pd.concat([local_climate_risk, sds_row, sud_row], ignore_index=True)

# Remove the original 'SDN' row
local_climate_risk = local_climate_risk[local_climate_risk['Country'] != 'SDN']

csv_file = "Other_data/country_codes.csv"
iso_convert = pd.read_csv(csv_file, delimiter=',')  # Assuming tab-delimited data
iso_convert.loc[iso_convert['name'] == 'Namibia', 'alpha-2'] = 'NA' # Namibia line poses problems because it is read as 'nan'.
iso_convert.loc[iso_convert['name'] == 'South Sudan', 'alpha-3'] = 'SDS'
iso_convert.loc[iso_convert['name'] == 'Sudan', 'alpha-3'] = 'SUD'

local_climate_risk = pd.merge(local_climate_risk, iso_convert[['name','alpha-2', 'alpha-3']], left_on='Country', right_on='alpha-3', how='left')

local_climate_risk = local_climate_risk[['name','Country','Local Climate Risk']]
local_climate_risk.rename(columns={'name': 'Country', 'Country': 'Country Code'}, inplace=True)

# Divinding countries according to a low / high climate risk threshold
#######

threshold = 50

low_risk_countries = local_climate_risk[local_climate_risk['Local Climate Risk'] >= threshold]['Country Code'].tolist()
high_risk_countries = local_climate_risk[local_climate_risk['Local Climate Risk'] < threshold]['Country Code'].tolist()


## Distributing EORA's countries by risk level
#########

# Importing EORA's countries (3-letter code)

L= mriot.L
country_list = L.index.get_level_values(0).unique()
sector_list = L.index.get_level_values(1).unique()
MRIOT_list = list(country_list).copy()

low_risk_countries = list(set(low_risk_countries) & set(MRIOT_list))
high_risk_countries = list(set(high_risk_countries) & set(MRIOT_list))
len(low_risk_countries)+len(high_risk_countries)

no_risk_countries = [
    "TWN",  # Taiwan (Officially known as the Republic of China)
    "HKG",  # Hong Kong
    "ROW"
]

new_data = {
    'Country': ['China, Hong Kong SAR', 'Taiwan'],
    'Country Code': ['HKG', 'TWN'],
    'Local Climate Risk': [np.nan] * 2,  # Creating a list of NaN values
}

# Creating a DataFrame from the new data
new_df = pd.DataFrame(new_data)

# Appending the new DataFrame to the existing one using concat
local_climate_risk = pd.concat([local_climate_risk, new_df], ignore_index=True)

In [17]:
local_climate_risk

,Country,Country Code,Local Climate Risk
0,Norway,NOR,75.023064
1,Finland,FIN,73.886537
2,Switzerland,CHE,72.512767
3,Denmark,DNK,71.923544
4,Singapore,SGP,71.506500
...,...,...,...
183,Chad,TCD,26.951176
184,South Sudan,SDS,32.845793
185,Sudan,SUD,32.845793
186,"China, Hong Kong SAR",HKG,NaN


## Comparing ICIO and ND-GAIN countries
----

In the data, ICIO has 67 countries : 66 countries + ROW.

In the data, ND-GAIN has 185 countries.

The countries in ICIO that do not have a risk index in ND-GAIN are (see code below) : 
- HKG - Hong-Kong
- TWN - Taiwan

Taiwan and Hong Kong have China's risk, and are added to ND-GAIN file.

In the end, resulting data frame should have 66 lines, 2 of which without a risk index.

In [9]:
import pandas as pd

#  ICIO country list
#####
L= mriot.L
country_list = L.index.get_level_values(0).unique()
sector_list = L.index.get_level_values(1).unique()

MRIOT_3_letters = list(country_list).copy()

#  ND-GAIN country list

file_path = 'ND_GAIN/resources/gain/gain.csv'
local_climate_risk = pd.read_csv(file_path)
# Selecting only the required columns and renaming them
local_climate_risk = local_climate_risk[['ISO3', '2021']]
local_climate_risk.columns = ['Country', 'Local Climate Risk']
local_climate_risk = local_climate_risk.dropna(subset=['Local Climate Risk']).reset_index(drop=True)
local_climate_risk = local_climate_risk.sort_values(by='Local Climate Risk', ascending=False)

ndgain_country_list = list(local_climate_risk['Country'])
icio_country_list = list(L.index.get_level_values(0).unique()) # 3 digits

# Comparing ND-GAIN with ICIO country lists
#####

# We want to know which countries of EORA do not have a risk index
icio_not_ndgain = list(set(icio_country_list) - set(ndgain_country_list))

print("Countries in ICIO with no risk index :", icio_not_ndgain)

Countries in ICIO with no risk index : ['TWN', 'ROW', 'HKG']


# Other data
---

Developed, developing, least developed country lists

Data come from UNTCAD : https://unctadstat.unctad.org/EN/Classifications.html.

In addition,
- Macau and Hong-Kong have been attributed to the same development group as Chine (developing),
- Monaco is considered developed,
- Virgin Islands (British) has been put in the same group as the United Kingdom (developed).

Names have been changed marginally to be coherent with country names used in other data.

In [1]:
import pickle

# Load the file
with open("Other_data/UN_developed_data.pkl", "rb") as file:
  lists_data = pickle.load(file)

# Retrieve the lists
least_developed_countries = lists_data["least_developed_countries"]
developed_countries = lists_data["developed_countries"]
developing_countries = lists_data["developing_countries"]

# Print the lists
print("Lists retrieved")

Lists retrieved


# Country-level analysis
----

## Imports interm. cons.
----

In [7]:
import pandas as pd

Y = mriot.Y
y = Y.sum(axis = 1)

L= mriot.L

country_list = L.index.get_level_values(0).unique()
sector_list = L.index.get_level_values(1).unique()


values = {'Low Risk': [0] * country_list.size}
cForeign_Inputs_low = pd.DataFrame(values, index=country_list)

values = {'High Risk': [0] * country_list.size}
cForeign_Inputs_high = pd.DataFrame(values, index=country_list)

values = {'ROW': [0] * country_list.size}
cForeign_Inputs_row = pd.DataFrame(values, index=country_list)

values = {'Foreign (%)': [0] * country_list.size}
cForeign_Inputs = pd.DataFrame(values, index=country_list)

values = {'Local (%)': [0] * country_list.size}
cLocal_Inputs = pd.DataFrame(values, index=country_list)


for c in country_list:
    intrants_c = 0
    intrants_c_foreign = 0
    intrants_c_foreign_low = 0
    intrants_c_foreign_high = 0
    intrants_c_foreign_row = 0

    
    for s in sector_list:
        L_cs = L[c,s]
        y_cs = y[c,s]
        L_cs_foreign = L_cs[L_cs.index.get_level_values('region') != c]
        L_cs_foreign_low = L_cs[~L_cs.index.get_level_values('region').isin([c]) & L_cs.index.get_level_values('region').isin(low_risk_countries)]
        L_cs_foreign_high = L_cs[~L_cs.index.get_level_values('region').isin([c]) & L_cs.index.get_level_values('region').isin(high_risk_countries)]
        L_cs_foreign_row = L_cs[~L_cs.index.get_level_values('region').isin([c]) & L_cs.index.get_level_values('region').isin(no_risk_countries)]
       
        intrants_cs = (L_cs*y_cs).sum()
        intrants_cs_foreign = (L_cs_foreign*y_cs).sum()
        intrants_cs_foreign_low = (L_cs_foreign_low*y_cs).sum()
        intrants_cs_foreign_high = (L_cs_foreign_high*y_cs).sum()
        intrants_cs_foreign_row = (L_cs_foreign_row*y_cs).sum()

        intrants_c=intrants_c + intrants_cs
        intrants_c_foreign = intrants_c_foreign + intrants_cs_foreign   
        intrants_c_foreign_low = intrants_c_foreign_low + intrants_cs_foreign_low 
        intrants_c_foreign_high = intrants_c_foreign_high + intrants_cs_foreign_high  
        intrants_c_foreign_row = intrants_c_foreign_row + intrants_cs_foreign_row  

    cForeign_Inputs.loc[c] = intrants_c_foreign / intrants_c*100
    cForeign_Inputs_low.loc[c] = intrants_c_foreign_low / intrants_c*100
    cForeign_Inputs_high.loc[c] = intrants_c_foreign_high / intrants_c *100
    cForeign_Inputs_row.loc[c] = intrants_c_foreign_row / intrants_c *100

    cLocal_Inputs.loc[c] = 100 - cForeign_Inputs.loc[c].iloc[0]
    
# List of DataFrames to concatenate
dataframes_to_concat = [cForeign_Inputs_low, cForeign_Inputs_high, cForeign_Inputs_row, cForeign_Inputs, cLocal_Inputs]

# Concatenate the DataFrames along columns (axis=1)
resulting_dataframe_country_inputs_leontief = pd.concat(dataframes_to_concat, axis=1)

# Sorting the data by 'Non-OECD' for better visualization
result = resulting_dataframe_country_inputs_leontief.sort_values('Foreign (%)', ascending=False).reset_index()

# Define the function to categorize development status
def categorize_local_risk(region):
    if region in low_risk_countries:
        return 'Low Risk'
    elif region in high_risk_countries:
        return 'High Risk'
    elif region in no_risk_countries:
        return 'Unkonwn'
    
result['Local Risk'] = result['region'].apply(categorize_local_risk)

# Merge the two DataFrames on 'region' column
result = result.merge(local_climate_risk, left_on='region', right_on='Country Code', how='left')

result['Country Code']=result['region']

path='Other_data/World_Bank_Regions.xlsx'
wb_regions = pd.read_excel(path)
result = result.merge(wb_regions, left_on='Country Code', right_on='Code', how='left')

path='Other_data/World_Bank_Pop.csv'
wb_pop = pd.read_csv(path, delimiter=';')  # If semicolon-separated
result = result.merge(wb_pop, left_on='Country Code', right_on='Country Code', how='left')

result = result.drop(columns=['Series Name','Series Code','region','Country Name','Lending category','Economy','Code'])
result.rename(columns={'2023 [YR2023]':'Population'},inplace=True)

# Convert the column to numeric
result['Population'] = pd.to_numeric(result['Population'], errors='coerce').fillna(0)

# Define the function to categorize development status
def categorize_development(region):
    if region in least_developed_countries:
        return 'Least Developed'
    elif region in developed_countries:
        return 'Developed'
    elif region in developing_countries:
        return 'Developing'
    else:
        return 'Unknown'
    
result['Development'] = result['Country'].apply(categorize_development)
result['Share of Risky Trade Partners (%)']=result['High Risk']/(result['High Risk']+result['Low Risk'])*100
result = result.rename(columns={'High Risk': 'High Risk Exposure (%)'})
result = result.rename(columns={'Low Risk': 'Low Risk Exposure (%)'})

result = result[['Country','Country Code','Local Climate Risk','Local Risk','Region','Income group','Population','Development', 'Low Risk Exposure (%)', 'High Risk Exposure (%)','ROW','Foreign (%)','Local (%)',  'Share of Risky Trade Partners (%)']]

# Data are added manually for South and North Sudan because of uncorresponding country codes + Taiwan

result.loc[result['Country'] == 'South Sudan', 'Region'] = 'Sub-Saharan Africa'
result.loc[result['Country'] == 'South Sudan', 'Income group'] = 'Low income'
result.loc[result['Country'] == 'South Sudan', 'Population'] = 11088796

result.loc[result['Country'] == 'Sudan', 'Region'] = 'Sub-Saharan Africa'
result.loc[result['Country'] == 'Sudan', 'Income group'] = 'Low income'
result.loc[result['Country'] == 'Sudan', 'Population'] = 48109006

#result.loc[result['Country Code'] == 'TWN', 'Population'] = 23923276

result=result.reset_index()
result = result.drop('index', axis=1)

result = result.dropna(subset=['Region'])

resulting_dataframe_country_imports_interm_leontief = result

C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2700189746.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '9.986827960742666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cForeign_Inputs.loc[c] = intrants_c_foreign / intrants_c*100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2700189746.py:57: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '6.906606346506912' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cForeign_Inputs_low.loc[c] = intrants_c_foreign_low / intrants_c*100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2700189746.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '2.3328576671595744' has dtype incompatible with int64, please explicitly cast to a compatible dt

In [ ]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/resulting_dataframe_country_imports_interm_leontief_ndgain_icio.pkl'

# Save the dictionary to a file
with open(file_path_pickle, 'wb') as f:
#    pickle.dump(resulting_dataframe_country_imports_interm_leontief, f)

In [9]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/resulting_dataframe_country_imports_interm_leontief_ndgain_icio.pkl'

# Load the dictionary from the file
with open(file_path_pickle, 'rb') as f:
    resulting_dataframe_country_imports_interm_leontief = pickle.load(f)

In [10]:
df=resulting_dataframe_country_imports_interm_leontief
median_high_risk_developed_inputs = df[df['Development'] == 'Developed']['High Risk Exposure (%)'].median()
median_high_risk_developed_inputs

0.8599934703438987

## Imports fin. cons.
----

In [12]:
import pandas as pd

x = mriot.x
final_production_by_use = mriot.Y.copy()
y = final_production_by_use.sum(axis = 1)

L = mriot.L

country_list = L.index.get_level_values(0).unique()
sector_list = L.index.get_level_values(1).unique()

multi_index = pd.MultiIndex.from_product([country_list, country_list, sector_list], names=['No risk country', 'Country', 'Sector'])

values = {'Low Risk': [0] * multi_index.size}
risk_cs_low = pd.DataFrame(values, index=multi_index)

values = {'High Risk': [0] * multi_index.size}
risk_cs_high = pd.DataFrame(values, index=multi_index)

values = {'ROW': [0] * multi_index.size}
risk_cs_row = pd.DataFrame(values, index=multi_index)

def indicator(element, lst):
    return 1 if element in lst else 0

for cc in country_list:   
    country_list_except = [x for x in country_list if x != cc]
    for c in country_list_except:
        for s in sector_list:
            low_risk_countries_except =  [x for x in low_risk_countries if x != cc] ## cc is not risky, but c is, if it is in high_risk_countries
            high_risk_countries_except =  [x for x in high_risk_countries if x != cc]

            risk_cs_low.loc[cc,c,s] = y.loc[c,s]*indicator(c,low_risk_countries)+y.loc[c,s]*L[c,s].loc[low_risk_countries_except].sum()
            risk_cs_high.loc[cc,c,s] = y.loc[c,s]*indicator(c,high_risk_countries)+y.loc[c,s]*L[c,s].loc[high_risk_countries_except].sum()
            risk_cs_row.loc[cc,c,s] = y.loc[c,s]*indicator(c,['ROW'])+y.loc[c,s]*L[c,s].loc[['ROW']].sum()
            
            total = y.loc[c,s]+y.loc[c,s]*L[c,s].sum()
            
            risk_cs_low.loc[cc,c,s] = risk_cs_low.loc[cc,c,s]/total*100
            risk_cs_high.loc[cc,c,s] = risk_cs_high.loc[cc,c,s]/total*100
            risk_cs_row.loc[cc,c,s] = risk_cs_row.loc[cc,c,s]/total*100

    # List of DataFrames to concatenate
    risk_cs = [risk_cs_low, risk_cs_high,risk_cs_row]

    # Concatenate the DataFrames along columns (axis=1)
    risk_cs = pd.concat(risk_cs,axis=1)
    
# Replace all NaN values with zeros
risk_cs = risk_cs.fillna(0)

C:\Users\delah\AppData\Local\Temp\ipykernel_14980\1341594425.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '60012.05237815306' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  risk_cs_low.loc[cc,c,s] = y.loc[c,s]*indicator(c,low_risk_countries)+y.loc[c,s]*L[c,s].loc[low_risk_countries_except].sum()
C:\Users\delah\AppData\Local\Temp\ipykernel_14980\1341594425.py:34: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '439.23263415404915' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  risk_cs_high.loc[cc,c,s] = y.loc[c,s]*indicator(c,high_risk_countries)+y.loc[c,s]*L[c,s].loc[high_risk_countries_except].sum()
C:\Users\delah\AppData\Local\Temp\ipykernel_14980\1341594425.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

In [ ]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/risk_cs_imports_fin_leontief_ndgain_icio.pkl'

# Save the dictionary to a file
#with open(file_path_pickle, 'wb') as f:
    pickle.dump(risk_cs, f)

In [11]:
import pickle
import pandas as pd


# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/risk_cs_imports_fin_leontief_ndgain_icio.pkl'

# Load the dictionary from the file
with open(file_path_pickle, 'rb') as f:
    risk_cs = pickle.load(f)

In [12]:
import pandas as pd

final_production_by_use = mriot.Y.copy()
y = final_production_by_use.sum(axis = 1) # final production of (c,s), whatever use

Y = final_production_by_use.T.groupby('region').sum().T

x = mriot.x # Gross output

L= mriot.L # Leontief matrix

country_list = L.index.get_level_values(0).unique()
sector_list = L.index.get_level_values(1).unique()


values = {'Low Risk': [0] * country_list.size}
cConsoFin_low = pd.DataFrame(values, index=country_list)

values = {'High Risk': [0] * country_list.size}
cConsoFin_high = pd.DataFrame(values, index=country_list)

values = {'ROW': [0] * country_list.size}
cConsoFin_row = pd.DataFrame(values, index=country_list)

values = {'Foreign (%)': [0] * country_list.size}
cConsoFin_Foreign = pd.DataFrame(values, index=country_list)

values = {'Local (%)': [0] * country_list.size}
cConsoFin_Local = pd.DataFrame(values, index=country_list)

for cc in country_list:
    Y_cc = Y[cc]
    Y_cc_tot = Y_cc.sum()
    country_list_except = [x for x in country_list if x != cc]

    for c in country_list_except:
        for s in sector_list:
            cConsoFin_low.loc[cc] = cConsoFin_low.loc[cc].iloc[0]+Y_cc.loc[c,s]*risk_cs.loc[cc,c,s]['Low Risk']/100
            cConsoFin_high.loc[cc] = cConsoFin_high.loc[cc].iloc[0]+Y_cc.loc[c,s]*risk_cs.loc[cc,c,s]['High Risk']/100
            cConsoFin_row.loc[cc] = cConsoFin_row.loc[cc].iloc[0]+Y_cc.loc[c,s]*risk_cs.loc[cc,c,s]['ROW']/100

    cConsoFin_Foreign.loc[cc] = cConsoFin_low.loc[cc].iloc[0] + cConsoFin_high.loc[cc].iloc[0] + cConsoFin_row.loc[cc].iloc[0]
    cConsoFin_Local.loc[cc] = Y_cc_tot - cConsoFin_Foreign.loc[cc].iloc[0]

    #Ratios
    cConsoFin_low.loc[cc]=cConsoFin_low.loc[cc].iloc[0]/Y_cc_tot*100
    cConsoFin_high.loc[cc]=cConsoFin_high.loc[cc].iloc[0]/Y_cc_tot*100
    cConsoFin_row.loc[cc]=cConsoFin_row.loc[cc].iloc[0]/Y_cc_tot*100

    cConsoFin_Local.loc[cc]=cConsoFin_Local.loc[cc].iloc[0]/Y_cc_tot*100
    cConsoFin_Foreign.loc[cc]=cConsoFin_Foreign.loc[cc].iloc[0]/Y_cc_tot*100
    
# List of DataFrames to concatenate
dataframes_to_concat = [cConsoFin_low, cConsoFin_high,cConsoFin_row, cConsoFin_Foreign, cConsoFin_Local]

# Concatenate the DataFrames along columns (axis=1)
resulting_dataframe_country_cons_leontief = pd.concat(dataframes_to_concat, axis=1)

# Sorting the data by 'Non-OECD' for better visualization
result = resulting_dataframe_country_cons_leontief.sort_values('Foreign (%)', ascending=False).reset_index()

# Define the function to categorize development status
def categorize_local_risk(region):
    if region in low_risk_countries:
        return 'Low Risk'
    elif region in high_risk_countries:
        return 'High Risk'
    else:
        return 'Unknown'
    
result['Local Risk'] = result['region'].apply(categorize_local_risk)

result=result[result['Local Risk']!='Unknown']

# Merge the two DataFrames on 'region' column
result = result.merge(local_climate_risk, left_on='region', right_on='Country Code', how='left')

path='Other_data/World_Bank_Regions.xlsx'
wb_regions = pd.read_excel(path)
result = result.merge(wb_regions, left_on='Country Code', right_on='Code', how='left')

path='Other_data/World_Bank_Pop.csv'
wb_pop = pd.read_csv(path, delimiter=';')  # If semicolon-separated
result = result.merge(wb_pop, left_on='Country Code', right_on='Country Code', how='left')

result = result.drop(columns=['Series Name','Series Code','region','Country Name','Lending category','Economy','Code'])
result.rename(columns={'2023 [YR2023]':'Population'},inplace=True)

# Convert the column to numeric
result['Population'] = pd.to_numeric(result['Population'], errors='coerce').fillna(0)

# Define the function to categorize development status
def categorize_development(region):
    if region in least_developed_countries:
        return 'Least Developed'
    elif region in developed_countries:
        return 'Developed'
    elif region in developing_countries:
        return 'Developing'
    else:
        return 'Unknown'

result['Development'] = result['Country'].apply(categorize_development)

result['Share of Risky Trade Partners (%)']=result['High Risk']/(result['High Risk']+result['Low Risk'])*100
result = result.rename(columns={'High Risk': 'High Risk Exposure (%)'})
result = result.rename(columns={'Low Risk': 'Low Risk Exposure (%)'})

result = result[['Country','Country Code','Local Climate Risk','Local Risk','Region','Income group','Population','Development', 'Low Risk Exposure (%)', 'High Risk Exposure (%)','ROW','Foreign (%)','Local (%)',  'Share of Risky Trade Partners (%)']]

# Data are added manually for South and North Sudan because of uncorresponding country codes

result.loc[result['Country'] == 'South Sudan', 'Region'] = 'Sub-Saharan Africa'
result.loc[result['Country'] == 'South Sudan', 'Income group'] = 'Low income'
result.loc[result['Country'] == 'South Sudan', 'Population'] = 11088796

result.loc[result['Country'] == 'Sudan', 'Region'] = 'Sub-Saharan Africa'
result.loc[result['Country'] == 'Sudan', 'Income group'] = 'Low income'
result.loc[result['Country'] == 'Sudan', 'Population'] = 48109006

result=result.reset_index()
result = result.drop('index', axis=1)

resulting_dataframe_country_imports_fin_leontief = result

C:\Users\delah\AppData\Local\Temp\ipykernel_27008\1485568099.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.92410114649138' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cConsoFin_low.loc[cc] = cConsoFin_low.loc[cc].iloc[0]+Y_cc.loc[c,s]*risk_cs.loc[cc,c,s]['Low Risk']/100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\1485568099.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.006763564396040367' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cConsoFin_high.loc[cc] = cConsoFin_high.loc[cc].iloc[0]+Y_cc.loc[c,s]*risk_cs.loc[cc,c,s]['High Risk']/100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\1485568099.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.00621500111

In [ ]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/resulting_dataframe_country_imports_fin_leontief_ndgain_icio.pkl'

# Save the dictionary to a file
with open(file_path_pickle, 'wb') as f:
#    pickle.dump(resulting_dataframe_country_imports_fin_leontief, f)

In [17]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/resulting_dataframe_country_imports_fin_leontief_ndgain_icio.pkl'

# Load the dictionary from the file
with open(file_path_pickle, 'rb') as f:
    resulting_dataframe_country_imports_fin_leontief= pickle.load(f)

In [18]:
df=resulting_dataframe_country_imports_fin_leontief
median_high_risk_developed_consumption = df[df['Development'] == 'Developed']['High Risk Exposure (%)'].median()

median_high_risk_developed_consumption

0.790657297588635

## Exports interm. cons
----

In [19]:
import pandas as pd

Y = mriot.Y.copy()
x = mriot.x
y = Y.sum(axis = 1)

L= mriot.L

country_list = L.index.get_level_values(0).unique()
sector_list = L.index.get_level_values(1).unique()


values = {'Low Risk': [0] * country_list.size}
cExports_low = pd.DataFrame(values, index=country_list)

values = {'High Risk': [0] * country_list.size}
cExports_high = pd.DataFrame(values, index=country_list)

values = {'ROW': [0] * country_list.size}
cExports_row = pd.DataFrame(values, index=country_list)

values = {'Foreign (%)': [0] * country_list.size}
cExports = pd.DataFrame(values, index=country_list)

values = {'Local (%)': [0] * country_list.size}
cLocal_Use = pd.DataFrame(values, index=country_list)

for c in country_list:
    exports_c = 0
    exports_c_low = 0
    exports_c_high = 0
    exports_c_row = 0

    y_c = y[~y.index.get_level_values('region').isin([c])]
    y_c_low = y[~y.index.get_level_values('region').isin([c]) & y.index.get_level_values('region').isin(low_risk_countries)]
    y_c_high = y[~y.index.get_level_values('region').isin([c]) & y.index.get_level_values('region').isin(high_risk_countries)]
    y_c_row = y[~y.index.get_level_values('region').isin([c]) & y.index.get_level_values('region').isin(no_risk_countries)]

    x_c= x.loc[c].sum().iloc[0]

    for s in sector_list:
        L_cs = L.loc[c,s]
        
        L_cs = L_cs[L_cs.index.get_level_values('region') != c]
        L_cs_foreign_low = L_cs[~L_cs.index.get_level_values('region').isin([c]) & L_cs.index.get_level_values('region').isin(low_risk_countries)]
        L_cs_foreign_high = L_cs[~L_cs.index.get_level_values('region').isin([c]) & L_cs.index.get_level_values('region').isin(high_risk_countries)]
        L_cs_foreign_row = L_cs[~L_cs.index.get_level_values('region').isin([c]) & L_cs.index.get_level_values('region').isin(no_risk_countries)]

        exports_cs = (L_cs*y_c).sum()
        exports_cs_low = (L_cs_foreign_low*y_c_low).sum()
        exports_cs_high = (L_cs_foreign_high*y_c_high).sum()
        exports_cs_row = (L_cs_foreign_row*y_c_row).sum()

        exports_c = exports_c + exports_cs
        exports_c_low = exports_c_low + exports_cs_low
        exports_c_high = exports_c_high + exports_cs_high
        exports_c_row = exports_c_row + exports_cs_row


    cExports.loc[c] = exports_c / x_c*100
    cExports_low.loc[c] = exports_c_low / x_c*100
    cExports_high.loc[c] = exports_c_high / x_c *100
    cExports_row.loc[c] = exports_c_row / x_c *100

    
    cLocal_Use.loc[c] = 100 - cExports.loc[c].iloc[0]
    
# List of DataFrames to concatenate
dataframes_to_concat = [cExports_low, cExports_high,cExports_row, cExports, cLocal_Use]

# Concatenate the DataFrames along columns (axis=1)
resulting_dataframe_country_exports_leontief_CI = pd.concat(dataframes_to_concat, axis=1)

# Sorting the data by 'Non-OECD' for better visualization
result = resulting_dataframe_country_exports_leontief_CI.sort_values('Foreign (%)', ascending=False).reset_index()

# Define the function to categorize development status
def categorize_local_risk(region):
    if region in low_risk_countries:
        return 'Low Risk'
    elif region in high_risk_countries:
        return 'High Risk'
    else:
        return 'Unknown'
    
result['Local Risk'] = result['region'].apply(categorize_local_risk)

# Merge the two DataFrames on 'region' column
result = result.merge(local_climate_risk, left_on='region', right_on='Country Code', how='left')

path='Other_data/World_Bank_Regions.xlsx'
wb_regions = pd.read_excel(path)
result = result.merge(wb_regions, left_on='Country Code', right_on='Code', how='left')

path='Other_data/World_Bank_Pop.csv'
wb_pop = pd.read_csv(path, delimiter=';')  # If semicolon-separated
result = result.merge(wb_pop, left_on='Country Code', right_on='Country Code', how='left')

result = result.drop(columns=['Series Name','Series Code','region','Country Name','Lending category','Economy','Code'])
result.rename(columns={'2023 [YR2023]':'Population'},inplace=True)

# Convert the column to numeric
result['Population'] = pd.to_numeric(result['Population'], errors='coerce').fillna(0)

# Define the function to categorize development status
def categorize_development(region):
    if region in least_developed_countries:
        return 'Least Developed'
    elif region in developed_countries:
        return 'Developed'
    elif region in developing_countries:
        return 'Developing'
    else:
        return 'Unknown'

result['Development'] = result['Country'].apply(categorize_development)

result['Share of Risky Trade Partners (%)']=result['High Risk']/(result['High Risk']+result['Low Risk'])*100
result = result.rename(columns={'High Risk': 'High Risk Exposure (%)'})
result = result.rename(columns={'Low Risk': 'Low Risk Exposure (%)'})

result = result[['Country','Country Code','Local Climate Risk','Local Risk','Region','Income group','Population','Development', 'Low Risk Exposure (%)', 'High Risk Exposure (%)','ROW','Foreign (%)','Local (%)',  'Share of Risky Trade Partners (%)']]

# Data are added manually for South and North Sudan because of uncorresponding country codes

result.loc[result['Country'] == 'South Sudan', 'Region'] = 'Sub-Saharan Africa'
result.loc[result['Country'] == 'South Sudan', 'Income group'] = 'Low income'
result.loc[result['Country'] == 'South Sudan', 'Population'] = 11088796

result.loc[result['Country'] == 'Sudan', 'Region'] = 'Sub-Saharan Africa'
result.loc[result['Country'] == 'Sudan', 'Income group'] = 'Low income'
result.loc[result['Country'] == 'Sudan', 'Population'] = 48109006

result=result.reset_index()
result = result.drop('index', axis=1)

result = result.dropna(subset=['Region'])

resulting_dataframe_country_exports_interm_leontief = result

C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2706811163.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '8.00766708635492' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cExports.loc[c] = exports_c / x_c*100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2706811163.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '4.988995785451576' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cExports_low.loc[c] = exports_c_low / x_c*100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2706811163.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '2.055201711151279' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cExports_high.loc[c] = exports_c_hi

In [ ]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/resulting_dataframe_country_exports_interm_leontief_ndgain_icio.pkl'

# Save the dictionary to a file
with open(file_path_pickle, 'wb') as f:
#    pickle.dump(resulting_dataframe_country_exports_interm_leontief, f)

In [21]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/resulting_dataframe_country_exports_interm_leontief_ndgain_icio.pkl'

# Load the dictionary from the file
with open(file_path_pickle, 'rb') as f:
    resulting_dataframe_country_exports_interm_leontief = pickle.load(f)

## Exports fin. cons.
----

In [22]:
import pandas as pd

final_production = mriot.Y.copy()

L= mriot.L

country_list = L.index.get_level_values(0).unique()
sector_list = L.index.get_level_values(1).unique()

multi_index = pd.MultiIndex.from_product([country_list, country_list, sector_list], names=['No risk country', 'Country', 'Sector'])

Y = mriot.Y.copy()
# Group by the first level ('region') and sum values along columns for each 'region'
final_production_by_destination = Y.groupby(level='region', axis=1).sum()

values = {'Low Risk': [0] * multi_index.size}
risk_cs_low = pd.DataFrame(values, index=multi_index)

values = {'High Risk': [0] * multi_index.size}
risk_cs_high = pd.DataFrame(values, index=multi_index)

values = {'ROW': [0] * multi_index.size}
risk_cs_row = pd.DataFrame(values, index=multi_index)

for cc in country_list:   
    country_list_except = [x for x in country_list if x != cc]
    for c in country_list_except:
        for s in sector_list:
            final_production_cs=final_production_by_destination.loc[c,s]
            final_production_cs_tot = final_production_cs.sum()
            if final_production_cs_tot>0:
                risk_cs_low.loc[cc,c,s] = final_production_cs[~final_production_cs.index.isin([cc])&final_production_cs.index.isin(low_risk_countries)].sum() / final_production_cs_tot*100
                risk_cs_high.loc[cc,c,s] = final_production_cs[~final_production_cs.index.isin([cc])&final_production_cs.index.isin(high_risk_countries)].sum() / final_production_cs_tot*100
                risk_cs_row.loc[cc,c,s] = final_production_cs[~final_production_cs.index.isin([cc])&final_production_cs.index.isin(no_risk_countries)].sum() / final_production_cs_tot*100

    # List of DataFrames to concatenate
    risk_cs = [risk_cs_low, risk_cs_high,risk_cs_row]

    # Concatenate the DataFrames along columns (axis=1)
    risk_cs = pd.concat(risk_cs,axis=1)

C:\Users\delah\AppData\Local\Temp\ipykernel_27008\140244954.py:14: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  final_production_by_destination = Y.groupby(level='region', axis=1).sum()
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\140244954.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '92.26812756122109' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  risk_cs_low.loc[cc,c,s] = final_production_cs[~final_production_cs.index.isin([cc])&final_production_cs.index.isin(low_risk_countries)].sum() / final_production_cs_tot*100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\140244954.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '3.423143012353668' has dtype incompatible with int64, please explicitly cast to a compatible dtype 

In [ ]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/risk_cs_exports_fin_leontief_ndgain_icio.pkl'

# Save the dictionary to a file
with open(file_path_pickle, 'wb') as f:
#    pickle.dump(risk_cs, f)

In [24]:
import pickle
import pandas as pd


# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/risk_cs_exports_fin_leontief_ndgain_icio.pkl'

# Load the dictionary from the file
with open(file_path_pickle, 'rb') as f:
    risk_cs = pickle.load(f)

In [25]:
import pandas as pd

final_production_by_use = mriot.Y.copy()
y = final_production_by_use.sum(axis = 1) # final production of (c,s), whatever use

x = mriot.x # Gross output

L= mriot.L # Leontief matrix

country_list = L.index.get_level_values(0).unique()
sector_list = L.index.get_level_values(1).unique()


values = {'Low Risk': [0] * country_list.size}
cExports_low = pd.DataFrame(values, index=country_list)

values = {'High Risk': [0] * country_list.size}
cExports_high = pd.DataFrame(values, index=country_list)

values = {'ROW': [0] * country_list.size}
cExports_row = pd.DataFrame(values, index=country_list)

values = {'Foreign (%)': [0] * country_list.size}
cExports = pd.DataFrame(values, index=country_list)

values = {'Local (%)': [0] * country_list.size}
cLocal_Use = pd.DataFrame(values, index=country_list)


for c in country_list:
    exports_c = 0
    exports_c_low = 0
    exports_c_high = 0
    exports_c_row = 0

    final_production = y
    final_production_and_risk_c_low = y*risk_cs.loc[c]['Low Risk']/100
    final_production_and_risk_c_high = y*risk_cs.loc[c]['High Risk']/100
    final_production_and_risk_c_row = y*risk_cs.loc[c]['ROW']/100

    for s in sector_list:
        L_cs = L.loc[c,s]
        
        exports_cs= (L_cs*final_production).sum()
        exports_cs_low = (L_cs*final_production_and_risk_c_low).sum()
        exports_cs_high = (L_cs*final_production_and_risk_c_high).sum()
        exports_cs_row = (L_cs*final_production_and_risk_c_row).sum()

        exports_c = exports_c + exports_cs
        exports_c_low = exports_c_low + exports_cs_low  
        exports_c_high = exports_c_high + exports_cs_high  
        exports_c_row = exports_c_row + exports_cs_row 


    x_c = x.loc[c].sum().iloc[0]

    cExports_low.loc[c] = exports_c_low / x_c*100
    cExports_high.loc[c] = exports_c_high / x_c *100
    cExports_row.loc[c] = exports_c_row / x_c *100
    
    cExports.loc[c] = cExports_low.loc[c].iloc[0]+cExports_high.loc[c].iloc[0]+cExports_row.loc[c].iloc[0]

    cLocal_Use.loc[c] = 100 - cExports_low.loc[c].iloc[0]- cExports_high.loc[c].iloc[0]- cExports_row.loc[c].iloc[0]
    
# List of DataFrames to concatenate
dataframes_to_concat = [cExports_low, cExports_high,cExports_row,cExports, cLocal_Use]

# Concatenate the DataFrames along columns (axis=1)
resulting_dataframe_country_exports_leontief_CF = pd.concat(dataframes_to_concat, axis=1)

# Sorting the data by 'Non-OECD' for better visualization
result = resulting_dataframe_country_exports_leontief_CF.sort_values('Foreign (%)', ascending=False).reset_index()


# Define the function to categorize development status
def categorize_local_risk(region):
    if region in low_risk_countries:
        return 'Low Risk'
    elif region in high_risk_countries:
        return 'High Risk'
    else:
        return 'Unknown'
    
result['Local Risk'] = result['region'].apply(categorize_local_risk)

# Merge the two DataFrames on 'region' column
result = result.merge(local_climate_risk, left_on='region', right_on='Country Code', how='left')

path='Other_data/World_Bank_Regions.xlsx'
wb_regions = pd.read_excel(path)
result = result.merge(wb_regions, left_on='Country Code', right_on='Code', how='left')

path='Other_data/World_Bank_Pop.csv'
wb_pop = pd.read_csv(path, delimiter=';')  # If semicolon-separated
result = result.merge(wb_pop, left_on='Country Code', right_on='Country Code', how='left')

result = result.drop(columns=['Series Name','Series Code','region','Country Name','Lending category','Economy','Code'])
result.rename(columns={'2023 [YR2023]':'Population'},inplace=True)

# Convert the column to numeric
result['Population'] = pd.to_numeric(result['Population'], errors='coerce').fillna(0)

# Define the function to categorize development status
def categorize_development(region):
    if region in least_developed_countries:
        return 'Least Developed'
    elif region in developed_countries:
        return 'Developed'
    elif region in developing_countries:
        return 'Developing'
    else:
        return 'Unknown'

result['Development'] = result['Country'].apply(categorize_development)

result['Share of Risky Trade Partners (%)']=result['High Risk']/(result['High Risk']+result['Low Risk'])*100

result = result.rename(columns={'High Risk': 'High Risk Exposure (%)'})
result = result.rename(columns={'Low Risk': 'Low Risk Exposure (%)'})

result = result[['Country','Country Code','Local Climate Risk','Local Risk','Region','Income group','Population','Development', 'Low Risk Exposure (%)', 'High Risk Exposure (%)','ROW','Foreign (%)','Local (%)',  'Share of Risky Trade Partners (%)']]

# Data are added manually for South and North Sudan because of uncorresponding country codes

result.loc[result['Country'] == 'South Sudan', 'Region'] = 'Sub-Saharan Africa'
result.loc[result['Country'] == 'South Sudan', 'Income group'] = 'Low income'
result.loc[result['Country'] == 'South Sudan', 'Population'] = 11088796

result.loc[result['Country'] == 'Sudan', 'Region'] = 'Sub-Saharan Africa'
result.loc[result['Country'] == 'Sudan', 'Income group'] = 'Low income'
result.loc[result['Country'] == 'Sudan', 'Population'] = 48109006

result=result.reset_index()
result = result.drop('index', axis=1)

result = result.dropna(subset=['Region'])

resulting_dataframe_country_exports_fin_leontief = result

C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2976250542.py:57: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '5.089118607875457' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cExports_low.loc[c] = exports_c_low / x_c*100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2976250542.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1.8812757335219286' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cExports_high.loc[c] = exports_c_high / x_c *100
C:\Users\delah\AppData\Local\Temp\ipykernel_27008\2976250542.py:59: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1.0125065386015277' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cExports_row.loc[c] =

In [26]:
resulting_dataframe_country_exports_fin_leontief

,Country,Country Code,Local Climate Risk,Local Risk,Region,Income group,Population,Development,Low Risk Exposure (%),High Risk Exposure (%),ROW,Foreign (%),Local (%),Share of Risky Trade Partners (%)
0,Luxembourg,LUX,68.424627,Low Risk,Europe & Central Asia,High income,6.575690e+05,Developed,46.362627,2.405280,4.682083,53.449990,46.550010,4.932096
1,Ireland,IRL,65.117169,Low Risk,Europe & Central Asia,High income,5.120455e+06,Developed,34.751346,2.311347,2.948071,40.010765,59.989235,6.236316
2,Singapore,SGP,71.506500,Low Risk,East Asia & Pacific,High income,5.673743e+06,Developing,27.909565,7.357491,4.268058,39.535115,60.464885,20.862221
3,Brunei Darussalam,BRN,57.181847,Low Risk,East Asia & Pacific,High income,4.525240e+05,Developing,23.963492,8.678383,4.261285,36.903161,63.096839,26.586657
4,Taiwan,TWN,NaN,Unknown,East Asia & Pacific,High income,2.392328e+07,Developed,26.965701,3.777909,2.515683,33.259292,66.740708,12.288436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,Brazil,BRA,48.899947,High Risk,Latin America & Caribbean,Upper middle income,2.164224e+08,Developing,8.206083,1.241876,1.004986,10.452946,89.547054,13.144387
63,India,IND,44.563042,High Risk,South Asia,Lower middle income,1.428628e+09,Developing,6.846254,0.825435,1.532238,9.203927,90.796073,10.759492
64,China,CHN,58.324542,Low Risk,East Asia & Pacific,Upper middle income,1.411878e+09,Developing,6.139918,1.425741,1.174162,8.739821,91.260179,18.844899
65,Argentina,ARG,49.617584,High Risk,Latin America & Caribbean,Upper middle income,4.651925e+07,Developing,5.089119,1.881276,1.012507,7.982901,92.017099,26.989517


In [ ]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/resulting_dataframe_country_exports_fin_leontief_ndgain_icio.pkl'

# Save the dictionary to a file
with open(file_path_pickle, 'wb') as f:
#    pickle.dump(resulting_dataframe_country_exports_fin_leontief, f)

In [28]:
import pickle

# Explicit file path
file_path_pickle = 'ND_GAIN/results/Country/resulting_dataframe_country_exports_fin_leontief_ndgain_icio.pkl'

# Load the dictionary from the file
with open(file_path_pickle, 'rb') as f:
    resulting_dataframe_country_exports_fin_leontief = pickle.load(f)